# UCI Bank Marketing : Applied Machine Learning: Model Design, Training, and Performance Evaluation

**Author:** Min Khaung Min Lwin

**Dataset:** Bank Marking (Bank_full.csv), UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/222/bank+marketing

## 1. Load and Inspect the Data (Task 2)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

df = pd.read_csv("bank/bank-full.csv", sep=";")   # the file is semicolon-separated
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum().sum(), "missing values (NaN)")
df.describe().T

(45211, 17)
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object
0 missing values (NaN)


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


In [3]:
# Target balance
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True).round(3))

# "unknown" is used instead of NaN for missing categories
cat_cols = df.select_dtypes(include=["object", "str"]).columns
print((df[cat_cols] == "unknown").sum().sort_values(ascending=False))

y
no     39922
yes     5289
Name: count, dtype: int64
y
no     0.883
yes    0.117
Name: proportion, dtype: float64
poutcome     36959
contact      13020
education     1857
job            288
marital          0
default          0
housing          0
loan             0
month            0
y                0
dtype: int64


In [4]:
# Two columns that need special handling
print((df["pdays"] == -1).mean().round(3), "share with pdays = -1 (not previously contacted)")
print(df.groupby("y")["duration"].median(), "<- median call duration by outcome")

0.817 share with pdays = -1 (not previously contacted)
y
no     164.0
yes    426.0
Name: duration, dtype: float64 <- median call duration by outcome


In [5]:
pd.crosstab(df["pdays"] == -1, df["poutcome"])

poutcome,failure,other,success,unknown
pdays,,,,
False,4901,1840,1511,5
True,0,0,0,36954


## 2. data preparation (Task 3)

### Preparation decisions

**Dropping `duration` (target leakage).** `duration` is the length of the last call in seconds. The median call lasted **426 s** for clients who subscribed and **164 s** for those who did not, so the column is very predictive. The problem is that the length of a call is only known once the call has ended, and by then the outcome is already known. A model trained with `duration` would score well in evaluation but could not be used to decide *whom to call*, which is the point of the model. The dataset documentation also says this column should be dropped for a realistic predictive model.

**Handling `pdays = -1` (a sentinel value, not a real number).** `pdays` is the number of days since the client was last contacted in a previous campaign, and **-1 means "never contacted"** (81.7% of rows). Left as it is, a linear model would treat -1 as "contacted one day *before* today", which makes no sense. The crosstab above shows that `pdays = -1` matches `poutcome = "unknown"` in **36,954 rows**, and never occurs together with `failure`, `other` or `success`. So "unknown" in `poutcome` really means "no previous campaign". To keep that information without the misleading number:
- a new binary feature `previously_contacted` (1 if `pdays != -1`) keeps the "contacted before or not" signal explicit;
- `pdays` is set to 0 for never-contacted clients, so the numeric column only carries meaning for clients who were contacted before, and the flag tells the model which case applies.

(Only 5 rows have `poutcome = "unknown"` with a real `pdays` value, so the two columns agree almost perfectly.)

**Stratified train/test split.** Only **11.7%** of clients subscribed. A purely random 80/20 split could, by chance, put a slightly different share of "yes" cases in the test set, which would make the test metrics less reliable. `stratify=y` keeps the positive rate at 0.117 in both the training and test sets (confirmed in the output). The test set (20%, 9,043 rows) is set aside now and is not used again until the final evaluation; all model selection below uses only the training set.

In [6]:
from sklearn.model_selection import train_test_split

data = df.drop(columns=["duration"]).copy()          # leakage: only known after the call
data["previously_contacted"] = (data["pdays"] != -1).astype(int)
data.loc[data["pdays"] == -1, "pdays"] = 0
data["y"] = (data["y"] == "yes").astype(int)

X = data.drop(columns=["y"])
y = data["y"]

# Stratified split keeps the 11.7% "yes" rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

(36168, 16) (9043, 16) 0.117 0.117


**Keeping `"unknown"` as its own category.** The dataset has no NaNs; missing categorical values are recorded as the string `"unknown"` (e.g. `contact` 13,020 rows, `education` 1,857, `job` 288). These values are **kept as a separate category** and not imputed or dropped, because:
- the missingness carries information. For `poutcome`, "unknown" means "never contacted before" (see above); for `contact`, almost all "unknown" rows come from May–June, and these clients subscribe at only ~4% versus ~15% for clients with a known contact type;
- dropping rows would discard up to ~30% of the data (the `contact` column alone), and filling in the most frequent value would invent information we do not have;
- one-hot encoding simply gives "unknown" its own indicator column, so the model can learn whether it matters.

**Encoding and scaling.** Categorical columns are one-hot encoded (`drop="if_binary"` keeps a single column for yes/no features; `handle_unknown="ignore"` protects against categories that appear only at prediction time). Numeric columns are standardised so that logistic regression is not dominated by large-scale features such as `balance`. Both steps sit inside a `ColumnTransformer` that goes into a `Pipeline`, so they are **fit only on the training folds** during cross-validation, which avoids leaking statistics from the validation folds.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_features = X.select_dtypes(include=["object", "str"]).columns.tolist()
num_features = X.select_dtypes(include="number").columns.tolist()

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features),
])

## 3. models and cross-validation (Task 4)

### Model choice and cross-validation setup

**Why these three models**
- **Baseline: `DummyClassifier` (always predicts "no").** Sets the minimum bar. With 88.3% "no", this model reaches about 88% accuracy without learning anything, so any real model has to clearly beat it on the metrics that matter for the "yes" class.
- **Logistic regression.** A simple, fast and interpretable linear model (coefficients show the direction of each feature's effect). It is a strong benchmark on tabular data and shows how much signal a linear decision boundary can capture.
- **Histogram gradient boosting.** A tree ensemble that captures non-linear effects and interactions (e.g. month × contact type, or age bands) without manual feature engineering. It is usually among the best-performing model families on tabular data and trains quickly on ~36k rows.

Both real models use `class_weight="balanced"`, which gives more weight to errors on the rare "yes" class so that the models do not simply copy the baseline.

**Cross-validation setup**
- **5-fold `StratifiedKFold`** (shuffled, `random_state=42`) on the **training set only**. Every fold keeps the 11.7% positive rate, and the test set stays untouched.
- The whole `Pipeline` (preprocessing + model) is cross-validated, so encoding and scaling are re-fit inside each fold.
- **Metrics:** accuracy is reported but is misleading with this class balance. The main metrics are **precision** (of the clients we would call, how many subscribe), **recall** (of all subscribers, how many we find), **F1** (balance of the two), and **average precision** (area under the precision–recall curve, which does not depend on a threshold; the baseline value equals the positive rate, 0.117).

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score


models = {
    "Baseline (always 'no')": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "average_precision": "average_precision",
}

cv_results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    cv_results[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

pd.DataFrame(cv_results).T.round(3)

,accuracy,precision,recall,f1,average_precision
Baseline (always 'no'),0.883,0.000,0.000,0.000,0.117
Logistic regression,0.755,0.266,0.623,0.373,0.398
Gradient boosting,0.825,0.358,0.616,0.452,0.443


### Interpreting the cross-validation results

- **Accuracy is misleading here.** The baseline has the *highest* accuracy (0.883) yet finds no subscribers at all (recall = 0, F1 = 0). The real models have lower accuracy because `class_weight="balanced"` makes them predict "yes" much more often. They accept more false positives in exchange for finding subscribers, which is the trade-off a marketing team wants.
- **Both real models clearly beat the baseline.** Average precision goes from 0.117 (chance level) to about 0.40–0.44, so the features carry real signal even without `duration`.
- **Gradient boosting is the strongest model.** At almost the same recall (~0.62 for both models), it has clearly better precision (0.358 vs 0.266), and therefore better F1 (0.452 vs 0.373) and average precision (0.443 vs 0.398). In practice, it finds the same share of subscribers while wasting fewer calls: about 1 in 3 clients it flags subscribes, compared with about 1 in 4 for logistic regression. The gap suggests that non-linear effects and interactions matter in this data.
- **Absolute performance is moderate.** Even the best model has precision of about 0.36, so most flagged clients still say "no". This is expected once the leaky `duration` feature is removed. It suggests that tuning gradient boosting and adjusting the decision threshold (to trade recall against precision according to the cost of a call) are the next steps, and gradient boosting is the candidate to carry forward to the final evaluation on the test set.